In [51]:
# Install ReportLab and DejaVuSans fonts for Cyrillic support
!pip install reportlab
!sudo apt-get update
!sudo apt-get install -y fonts-dejavu-core

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-dejavu-core is already the newest version (2.37-2build1).
0 upgraded, 0 newly installed,

In [61]:
import xml.etree.ElementTree as ET
from urllib.error import URLError, HTTPError
from tqdm.auto import tqdm

from Bio import Entrez

def get_clinvar_significance(rs_id):
    """
    Принимает rs-номер (например, 'rs1137282'),
    обращается к NCBI ClinVar и возвращает Clinical Significance.
    """
    Entrez.email = "genomic.atlas.project@example.com"  # Укажи любую почту

    try:
        # Шаг 1: Ищем ID в базе ClinVar по rs-номеру
        search_query = f"{rs_id}[Variant ID]"
        handle = Entrez.esearch(db="clinvar", term=search_query)
        search_results = Entrez.read(handle)
        handle.close()
    except (URLError, HTTPError) as e:
        return f"Network error during search: {e}"

    if not search_results['IdList']:
        return "No ClinVar record found."

    try:
        # Шаг 2: Получаем детали записи
        clinvar_id = search_results['IdList'][0]
        handle = Entrez.esummary(db="clinvar", id=clinvar_id)
        summary = Entrez.read(handle, validate=False)
        handle.close()
    except (URLError, HTTPError) as e:
        return f"Network error during summary retrieval: {e}"

    # Отладка: выводим структуру summary
    # print("Debug: Summary structure for rs_id", rs_id, ":\n", summary) # Закомментируем отладочный вывод для чистоты

    # Извлекаем клиническую значимость
    try:
        # В структуре ClinVar значение лежит в поле 'germline_classification' -> 'description'
        significance = summary['DocumentSummarySet']['DocumentSummary'][0]['germline_classification']['description']
        return significance
    except (KeyError, IndexError):
        return "Significance data unavailable."

# --- Оптимизированная внутренняя функция для пакетной обработки ---
def _get_clinvar_significance_batch_internal(rs_ids_list):
    """
    Внутренняя оптимизированная функция для пакетного получения клинической значимости.
    """
    Entrez.email = "genomic.atlas.project@example.com" # Убедиться, что email установлен
    final_results = {}
    clinvar_ids_to_fetch = []
    clinvar_id_map = {} # Сопоставление ClinVar ID с исходным rs_id

    # Шаг 1: Выполняем esearch для каждого rs_id, чтобы получить ClinVar ID
    for rs_id in tqdm(rs_ids_list, desc="Поиск ClinVar ID"):
        try:
            search_query = f"{rs_id}[Variant ID]"
            handle = Entrez.esearch(db="clinvar", term=search_query)
            search_results = Entrez.read(handle)
            handle.close()
        except (URLError, HTTPError) as e:
            final_results[rs_id] = f"Сетевая ошибка при поиске {rs_id}: {e}"
            continue

        if search_results['IdList']:
            clinvar_id = search_results['IdList'][0]
            clinvar_ids_to_fetch.append(clinvar_id)
            clinvar_id_map[clinvar_id] = rs_id
        else:
            final_results[rs_id] = "Запись ClinVar не найдена."

    # Шаг 2: Выполняем один запрос esummary для всех собранных ClinVar ID
    if clinvar_ids_to_fetch:
        print(f"DEBUG (batch): IDs to fetch for esummary: {clinvar_ids_to_fetch}") # Added debug
        id_list_str = ",".join(clinvar_ids_to_fetch)
        try:
            handle = Entrez.esummary(db="clinvar", id=id_list_str, validate=False)
            summary_list = Entrez.read(handle, validate=False)
            handle.close()
            if 'DocumentSummarySet' in summary_list and 'DocumentSummary' in summary_list['DocumentSummarySet']:
                print(f"DEBUG (batch): Number of DocumentSummary items received: {len(summary_list['DocumentSummarySet']['DocumentSummary'])}") # Added debug
            else:
                print(f"DEBUG (batch): No DocumentSummarySet or DocumentSummary found in esummary result for IDs: {id_list_str}")

        except (URLError, HTTPError) as e:
            for clinvar_id in clinvar_ids_to_fetch:
                rs_id = clinvar_id_map[clinvar_id]
                final_results[rs_id] = f"Сетевая ошибка при получении сводки для {rs_id}: {e}"
            return final_results

        for doc_summary in summary_list['DocumentSummarySet']['DocumentSummary']:
            # Correctly extract ClinVar ID from attributes if present
            clinvar_id_from_summary = doc_summary.get('Id')
            if clinvar_id_from_summary is None:
                clinvar_id_from_summary = doc_summary.get('uid') # Fallback to uid if Id not top-level
            # Check for 'attributes' as an attribute of DictElement and then get 'uid'
            if clinvar_id_from_summary is None and hasattr(doc_summary, 'attributes'):
                clinvar_id_from_summary = doc_summary.attributes.get('uid')

            print(f"DEBUG (batch): Processing summary for ClinVar ID: {clinvar_id_from_summary}") # Added debug

            if clinvar_id_from_summary is None:
                print("DEBUG (batch): ClinVar ID from summary is None, skipping entry.") # Added debug
                # Removed the 'Full doc_summary causing ID extraction issue' print as we now know the cause.
                continue

            original_rs_id = clinvar_id_map.get(str(clinvar_id_from_summary))
            print(f"DEBUG (batch): Mapped ClinVar ID {clinvar_id_from_summary} to original_rs_id: {original_rs_id}") # Added debug

            if original_rs_id:
                try:
                    significance = doc_summary['germline_classification']['description']
                    final_results[original_rs_id] = significance
                    print(f"DEBUG (batch): Successfully added {original_rs_id}: {significance}") # Added debug
                except (KeyError, IndexError):
                    final_results[original_rs_id] = "Данные о значимости недоступны."
                    print(f"DEBUG (batch): Significance data unavailable for {original_rs_id}.") # Added debug
            else:
                print(f"DEBUG (batch): Original rs_id not found in map for ClinVar ID {clinvar_id_from_summary}.") # Added debug

    # Fallback: Ensure every rs_id from the original list has an entry in final_results
    for rs_id in rs_ids_list:
        if rs_id not in final_results:
            final_results[rs_id] = "Не удалось получить данные (неизвестная ошибка или пропущено)."
            print(f"DEBUG (batch): Fallback applied for {rs_id}.") # Added debug

    print(f"DEBUG: Final results before returning: {final_results}") # Original debug print
    return final_results

# --- Оболочка для сохранения существующего имени функции ---
def get_clinvar_significance_for_list(rs_ids_list):
    """
    Принимает список rs-номеров и возвращает словарь с Clinical Significance для каждого,
    используя оптимизированные пакетные запросы.
    """
    return _get_clinvar_significance_batch_internal(rs_ids_list)


def load_rs_from_file(filepath):
    """
    Загружает список rs-номеров из текстового файла.
    Предполагается, что каждый rs-номер находится на новой строке.
    """
    rs_ids = []
    try:
        with open(filepath, 'r') as f:
            for line in f:
                stripped_line = line.strip()
                if stripped_line: # Проверка на пустые строки
                    rs_ids.append(stripped_line)
        return rs_ids
    except FileNotFoundError:
        print(f"Ошибка: Файл не найден по пути: {filepath}")
        return []
    except Exception as e:
        print(f"Произошла ошибка при чтении файла: {e}")
        return []

def add_rs_to_file(filepath, rs_ids_to_add):
    """
    Добавляет новые rs-номера в текстовый файл.
    Каждый rs-номер будет добавлен на новую строку, если его еще нет в файле.
    """
    existing_rs_ids = []
    try:
        with open(filepath, 'r') as f:
            existing_rs_ids = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        print(f"Файл {filepath} не найден. Будет создан новый.")

    new_entries_added = 0
    with open(filepath, 'a') as f:
        for rs_id in rs_ids_to_add:
            if rs_id not in existing_rs_ids:
                f.write(f"{rs_id}\n")
                new_entries_added += 1
                existing_rs_ids.append(rs_id) # Добавляем в список, чтобы избежать дублирования в рамках одного вызова
    print(f"Добавлено {new_entries_added} новых rs-номеров в файл {filepath}.")
    if new_entries_added == 0:
        print("Все предоставленные rs-номера уже существуют в файле.")

# Пример: проверка с одним rs-номером
print(f"ClinVar Status for rs1137282: {get_clinvar_significance('rs1137282')}")

# Пример: проверка со списком rs-номеров
rs_numbers_to_check = ['rs1137282', 'rs7069102', 'rs1234567'] # rs7069102 ранее не находился, rs1234567 - вымышленный для теста
list_results = get_clinvar_significance_for_list(rs_numbers_to_check)
print("\nClinVar Status for list of rs-numbers:")
for rs_id, significance in list_results.items():
    print(f"  {rs_id}: {significance}")

ClinVar Status for rs1137282: Benign


Поиск ClinVar ID:   0%|          | 0/3 [00:00<?, ?it/s]

DEBUG (batch): IDs to fetch for esummary: ['41449']
DEBUG (batch): Number of DocumentSummary items received: 1
DEBUG (batch): Processing summary for ClinVar ID: 41449
DEBUG (batch): Mapped ClinVar ID 41449 to original_rs_id: rs1137282
DEBUG (batch): Successfully added rs1137282: Benign
DEBUG: Final results before returning: {'rs7069102': 'Запись ClinVar не найдена.', 'rs1234567': 'Запись ClinVar не найдена.', 'rs1137282': 'Benign'}

ClinVar Status for list of rs-numbers:
  rs7069102: Запись ClinVar не найдена.
  rs1234567: Запись ClinVar не найдена.
  rs1137282: Benign


In [56]:
import re  # Стандартная библиотека для поиска текста
from Bio import Entrez  # Библиотека для работы с биологическими БД
# from clinvar_api import get_clinvar_significance_for_list, load_rs_from_file  # Удаляем этот импорт
# !pip install fpdf2 # Убедимся, что fpdf2 установлен
# from fpdf2 import FPDF # Изменено на fpdf2

def universal_genomic_scanner(sequence):
    print("\n" + "="*45)
    print("🚀 STARTING MULTI-LEVEL GENOMIC SCAN v3.0")
    print("="*45)

    # 1. Расчет GC-состава
    gc = (sequence.count('G') + sequence.count('C')) / len(sequence) * 100
    print(f"📊 [STRUCTURAL] GC-Content: {gc:.2f}%")

    # 2. ПРОВЕРКА ПОВТОРОВ (HTT)
    repeats = re.findall(r'(?:CAG){3,}', sequence)
    if repeats:
        count = len(max(repeats, key=len)) // 3
        status = "⚠️ PATHOGENIC" if count >= 36 else "✅ NORMAL"
        print(f"🧬 [REPEATS] HTT Gene: {count} CAG repeats ({status})")
    else:
        print(f"🧬 [REPEATS] HTT Gene: No expansions detected (Normal)")

    # 3. ПРОВЕРКА SIRT1 (Долголетие)
    if sequence[0] == 'G':
        print("🌟 [BIOHACK] SIRT1: Longevity Variant 'G' detected!")
        print("🔍 Connecting to ClinVar to verify...")
        clinical_status = get_clinvar_significance('rs7069102')
        print(f"   └─ Official Scientific Status: {clinical_status}")

    # 4. ПРОВЕРКА LRP5 (Кости-титан)
    if len(sequence) > 171 and sequence[171] == 'T':
        print("🦾 [SUPERPOWER] LRP5: High Bone Density variant found!")
        print("🔍 Verifying Clinical Significance...")
        clinical_status = get_clinvar_significance('rs121908675')
        print(f"   └─ ClinVar Result: {clinical_status}")

    # 5. ПРОВЕРКА TERT (Кнопка рака)
    if len(sequence) > 228 and sequence[228] == 'T':
        print("🚨 [CANCER RISK] TERT Promoter Mutation Found (C228T)")

    print("="*45 + "\n")

# ТЕСТОВЫЕ ДАННЫЕ (Полная проверка всех систем)
# Добавили CAG-повторы, чтобы пункт 2 тоже сработал
super_human_dna = "G" + ("CAG" * 40) + ("A" * 48) + "T" + ("A" * 56) + "T" + ("C" * 20)
universal_genomic_scanner(super_human_dna)


🚀 STARTING MULTI-LEVEL GENOMIC SCAN v3.0
📊 [STRUCTURAL] GC-Content: 40.89%
🧬 [REPEATS] HTT Gene: 40 CAG repeats (⚠️ PATHOGENIC)
🌟 [BIOHACK] SIRT1: Longevity Variant 'G' detected!
🔍 Connecting to ClinVar to verify...
   └─ Official Scientific Status: No ClinVar record found.



In [60]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from google.colab import files

# Register DejaVuSans fonts for Cyrillic support
# These fonts are typically available in Colab environments
pdfmetrics.registerFont(TTFont('DejaVuSans', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'))
pdfmetrics.registerFont(TTFont('DejaVuSans-Bold', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'))

def save_report_to_pdf(results, filename="Genomic_Report.pdf"):
    """Создает PDF-файл на основе словаря с результатами, используя ReportLab."""
    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter

    c.setFont("DejaVuSans-Bold", 16)
    c.drawCentredString(width / 2.0, height - 50, "Genomic Anomaly Atlas: Diagnostic Report")

    y_position = height - 100
    c.setFont("DejaVuSans-Bold", 12)
    c.drawString(50, y_position, "Variant (ID)")
    c.drawString(200, y_position, "Clinical Significance")
    y_position -= 20

    c.setFont("DejaVuSans", 10)
    for rs, status in results.items():
        if y_position < 70:  # Check if we need a new page
            c.showPage()
            y_position = height - 50
            c.setFont("DejaVuSans-Bold", 12)
            c.drawString(50, y_position, "Variant (ID)")
            c.drawString(200, y_position, "Clinical Significance")
            y_position -= 20
            c.setFont("DejaVuSans", 10)

        c.drawString(50, y_position, rs.upper())
        c.drawString(200, y_position, str(status))
        y_position -= 15

    c.save()
    print(f"📄 PDF Report saved as: {filename}")

def generate_professional_report(input_file):
    """
    Массовая проверка мутаций из файла и автоматическое создание PDF.
    """
    print("\n" + "═"*60)
    print("📋 GENOMIC ANOMALY ATLAS: BATCH PROCESSING")
    print("═"*60)

    # 1. Загружаем список rs-номеров из файла
    rs_list = load_rs_from_file(input_file)
    if not rs_list:
        print("❌ Error: No variants found in the input file.")
        return

    # 2. Получаем данные из ClinVar (пакетный запрос)
    print(f"🔍 Fetching data for {len(rs_list)} variants...")
    results = get_clinvar_significance_for_list(rs_list)

    # 3. Вывод в консоль для быстрой проверки
    for rs, status in results.items():
        print(f" -> {rs.upper()}: {status}")

    # 4. ВЫЗОВ PDF ГЕНЕРАТОРА (та самая 'основная логика')
    save_report_to_pdf(results)
    files.download('Genomic_Report.pdf')

    print("═"*60 + "\n")

# --- ГЛАВНЫЙ ЗАПУСК ---
if __name__ == "__main__":
    # Тест сканера сырой ДНК
    test_dna = "G" + ("CAG" * 40) + ("A" * 48) + "T" + ("A" * 56) + "T" + ("C" * 20)
    universal_genomic_scanner(test_dna)

    # Для демонстрации и тестирования, сначала обновим файл rs_numbers.txt
    with open("rs_numbers.txt", "w") as f:
        f.write("rs1137282\nrs7069102\nrs121908675\nrs1000000\nrs2000000\nrs3000000\n")

    # Тест пакетной обработки и генерации PDF
    # Убедись, что файл rs_numbers.txt лежит в той же папке в Colab!
    generate_professional_report("rs_numbers.txt")


🚀 STARTING MULTI-LEVEL GENOMIC SCAN v3.0
📊 [STRUCTURAL] GC-Content: 40.89%
🧬 [REPEATS] HTT Gene: 40 CAG repeats (⚠️ PATHOGENIC)
🌟 [BIOHACK] SIRT1: Longevity Variant 'G' detected!
🔍 Connecting to ClinVar to verify...
   └─ Official Scientific Status: No ClinVar record found.


════════════════════════════════════════════════════════════
📋 GENOMIC ANOMALY ATLAS: BATCH PROCESSING
════════════════════════════════════════════════════════════
🔍 Fetching data for 6 variants...


Поиск ClinVar ID:   0%|          | 0/6 [00:00<?, ?it/s]

DEBUG (batch): IDs to fetch for esummary: ['41449', '6216']
DEBUG (batch): Number of DocumentSummary items received: 2
DEBUG (batch): Processing summary for ClinVar ID: 41449
DEBUG (batch): Mapped ClinVar ID 41449 to original_rs_id: rs1137282
DEBUG (batch): Successfully added rs1137282: Benign
DEBUG (batch): Processing summary for ClinVar ID: 6216
DEBUG (batch): Mapped ClinVar ID 6216 to original_rs_id: rs121908675
DEBUG (batch): Successfully added rs121908675: Pathogenic/Likely pathogenic
DEBUG: Final results before returning: {'rs7069102': 'Запись ClinVar не найдена.', 'rs1000000': 'Запись ClinVar не найдена.', 'rs2000000': 'Запись ClinVar не найдена.', 'rs3000000': 'Запись ClinVar не найдена.', 'rs1137282': 'Benign', 'rs121908675': 'Pathogenic/Likely pathogenic'}
 -> RS7069102: Запись ClinVar не найдена.
 -> RS1000000: Запись ClinVar не найдена.
 -> RS2000000: Запись ClinVar не найдена.
 -> RS3000000: Запись ClinVar не найдена.
 -> RS1137282: Benign
 -> RS121908675: Pathogenic/Likel

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

════════════════════════════════════════════════════════════



Для создания файла `requirements.txt`, который содержит список внешних библиотек, используемых в вашем проекте, можно использовать следующую команду. Этот файл пригодится для воспроизводимости окружения проекта.

Мы включим в него:
*   `reportlab` (для генерации PDF)
*   `biopython` (для работы с Entrez)
*   `tqdm` (для индикаторов прогресса)

In [62]:
%%writefile requirements.txt
reportlab
biopython
tqdm

Writing requirements.txt
